# Visualizations — Seasonal NYC Airbnb Price Prediction

**Playground: seasonal_approach**

Six publication-ready plots for the progress report, all built with Plotly.
Static PNGs exported to `playground/seasonal_approach/plots/`.

| # | Plot | Source |
|---|------|--------|
| 1 | Seasonal demand index | `seasonal_index.csv` |
| 2 | Price distribution by month | `stacked_features.csv` |
| 3 | NYC listing map (price by location) | `feature_matrix.csv` |
| 4 | Price by borough & room type | `feature_matrix.csv` |
| 5 | SHAP feature importance | `shap_summary.csv` |
| 6 | Predicted vs actual (Nov test) | model + stacked data |


In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import xgboost as xgb
from pathlib import Path

DATA  = Path('data')
PLOTS = Path('plots')
PLOTS.mkdir(exist_ok=True)

# Shared style
FONT   = 'Inter, Arial, sans-serif'
BG     = '#FAFAFA'
PAPER  = '#FFFFFF'
ACCENT = '#FF5A5F'        # Airbnb red
BLUE   = '#00A699'        # Airbnb teal
GRAY   = '#767676'

def save(fig, name, w=900, h=500):
    fig.update_layout(
        font_family=FONT,
        plot_bgcolor=BG, paper_bgcolor=PAPER,
        margin=dict(l=60, r=40, t=60, b=60),
    )
    fig.write_image(str(PLOTS / f'{name}.png'), width=w, height=h, scale=2)
    fig.write_html(str(PLOTS / f'{name}.html'))
    print(f'  saved: plots/{name}.png + .html')

# Load data once
sf  = pd.read_csv(DATA / 'stacked_features.csv')
fm  = pd.read_csv(DATA / 'feature_matrix.csv')
si  = pd.read_csv(DATA / 'seasonal_index.csv')
shp = pd.read_csv(DATA / 'shap_summary.csv')

print('Data loaded:')
print(f'  stacked_features: {sf.shape}')
print(f'  feature_matrix:   {fm.shape}')
print(f'  seasonal_index:   {si.shape}')
print(f'  shap_summary:     {shp.shape}')


Data loaded:
  stacked_features: (28244, 64)
  feature_matrix:   (4073, 66)
  seasonal_index:   (12, 3)
  shap_summary:     (59, 2)


---
## Plot 1 — Seasonal demand index

In [2]:
MONTH_NAMES = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

colors = [ACCENT if m in si['month'].values else GRAY for m in range(1, 13)]
# training months get accent, others gray
train_months = set(range(4, 12))
bar_colors = [ACCENT if m in train_months else GRAY for m in si['month']]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=[MONTH_NAMES[m-1] for m in si['month']],
    y=si['seasonal_index'],
    marker_color=bar_colors,
    text=[f'{v:.3f}x' for v in si['seasonal_index']],
    textposition='outside',
    textfont_size=11,
))

fig.add_hline(y=1.0, line_dash='dash', line_color=GRAY,
              annotation_text='Annual average', annotation_position='right')

fig.update_layout(
    title=dict(text='NYC Airbnb Seasonal Demand Index (Review Volume Proxy)',
               font_size=16),
    xaxis_title='Month',
    yaxis_title='Demand Index (1.0 = annual average)',
    yaxis_range=[0, 1.45],
    showlegend=False,
)

save(fig, '01_seasonal_index')
fig.show()


  saved: plots/01_seasonal_index.png + .html


---
## Plot 2 — Price distribution by month

In [3]:
month_labels = {4:'Apr',5:'May',6:'Jun',7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov'}
sf['month_label'] = sf['month'].map(month_labels)
month_order = ['Apr','May','Jun','Jul','Aug','Sep','Oct','Nov']

# Cap display at 99th percentile for readability
p99 = sf['price_numeric'].quantile(0.99)
sf_plot = sf[sf['price_numeric'] <= p99].copy()

fig = px.box(
    sf_plot,
    x='month_label', y='price_numeric',
    category_orders={'month_label': month_order},
    color='month_label',
    color_discrete_sequence=px.colors.sequential.Teal,
    labels={'month_label': 'Month', 'price_numeric': 'Nightly Price ($)'},
    title='NYC STR Nightly Price Distribution by Month (Apr–Nov 2025)',
)

fig.update_traces(showlegend=False)
fig.update_layout(title_font_size=16)

save(fig, '02_price_by_month')
fig.show()


  saved: plots/02_price_by_month.png + .html


---
## Plot 3 — NYC listing map (price by location)

In [4]:
PRICE_CAP = 2000   # same cap as 02_stacking.ipynb — removes $50K placeholder prices

fm_plot = fm[fm['price_numeric'] <= PRICE_CAP].copy()

fig = px.scatter_map(
    fm_plot,
    lat='latitude', lon='longitude',
    color='price_numeric',
    color_continuous_scale='RdYlGn_r',
    size_max=6,
    zoom=10,
    center={'lat': 40.730, 'lon': -73.935},
    opacity=0.7,
    labels={'price_numeric': 'Nightly Price ($)'},
    title=f'NYC Short-Term Rental Listings — November 2025 ({len(fm_plot):,} STR listings)',
    hover_data={'latitude': False, 'longitude': False,
                'price_numeric': ':$.0f',
                'neighbourhood_cleansed': True},
)

fig.update_layout(
    title_font_size=16,
    coloraxis_colorbar=dict(title='Price ($)', tickprefix='$'),
    map_style='carto-positron',
)

save(fig, '03_listing_map', w=1000, h=600)
fig.show()

  saved: plots/03_listing_map.png + .html


---
## Plot 4 — Median price by borough & room type

In [5]:
ROOM_MAP = {0: 'Shared room', 1: 'Hotel room', 2: 'Private room', 3: 'Entire home/apt'}
fm_clean = fm[fm['price_numeric'] <= PRICE_CAP].copy()
fm_clean['room_type_label'] = fm_clean['room_type_ord'].map(ROOM_MAP)
fm_clean['borough'] = fm_clean['neighbourhood_group_cleansed']

grouped = (fm_clean.groupby(['borough', 'room_type_label'])['price_numeric']
             .median().reset_index()
             .rename(columns={'price_numeric': 'median_price'}))

boro_order = ['Manhattan', 'Brooklyn', 'Queens', 'Bronx', 'Staten Island']
room_order = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']

fig = px.bar(
    grouped,
    x='borough', y='median_price',
    color='room_type_label',
    barmode='group',
    category_orders={'borough': boro_order, 'room_type_label': room_order},
    color_discrete_sequence=[ACCENT, BLUE, '#FFB400', GRAY],
    labels={'borough': 'Borough', 'median_price': 'Median Nightly Price ($)',
            'room_type_label': 'Room Type'},
    title='Median Nightly Price by Borough and Room Type (November 2025)',
    text_auto='$.0f',
)

fig.update_traces(textposition='outside', textfont_size=9)
fig.update_layout(title_font_size=16, legend_title_text='Room Type',
                  yaxis_tickprefix='$')

save(fig, '04_price_by_borough_roomtype')
fig.show()

  saved: plots/04_price_by_borough_roomtype.png + .html


---
## Plot 5 — SHAP feature importance

In [6]:
top_n = 20
shp_top = shp.head(top_n).sort_values('mean_abs_shap')  # ascending for horizontal bar

# Category color coding
def feat_category(name):
    if name in ('month', 'seasonal_index'):           return 'Seasonal'
    if name.startswith('boro_'):                       return 'Borough'
    if name.startswith('prop_'):                       return 'Property type'
    if 'subway' in name or 'lirr' in name:             return 'Transit'
    if 'poi' in name or 'crime' in name:               return 'Spatial (POI/crime)'
    if name in ('is_superhost','host_response_rate_f',
                'host_acceptance_rate_f','host_response_time_ord',
                'host_identity_verified_f','has_license'):  return 'Host'
    if name.startswith('has_') or name == 'amenity_count':  return 'Amenities'
    if 'review' in name:                               return 'Reviews'
    return 'Property'

CAT_COLORS = {
    'Seasonal':         ACCENT,
    'Borough':          '#8B5CF6',
    'Property type':    BLUE,
    'Transit':          '#F59E0B',
    'Spatial (POI/crime)': '#10B981',
    'Host':             '#3B82F6',
    'Amenities':        '#EC4899',
    'Reviews':          '#6B7280',
    'Property':         '#1F2937',
}

shp_top['category'] = shp_top['feature'].apply(feat_category)
shp_top['color'] = shp_top['category'].map(CAT_COLORS)

fig = go.Figure()

for cat, grp in shp_top.groupby('category'):
    fig.add_trace(go.Bar(
        x=grp['mean_abs_shap'],
        y=grp['feature'],
        orientation='h',
        name=cat,
        marker_color=CAT_COLORS[cat],
    ))

fig.update_layout(
    title=dict(text=f'Top {top_n} Features by Mean |SHAP| Value (November Test Set)',
               font_size=16),
    xaxis_title='Mean |SHAP| (impact on log price prediction)',
    yaxis_title='',
    barmode='overlay',
    legend_title_text='Feature category',
    height=600,
    yaxis=dict(tickfont_size=11),
)

save(fig, '05_shap_importance', h=600)
fig.show()


  saved: plots/05_shap_importance.png + .html


---
## Plot 6 — Predicted vs actual (November test set)

In [7]:
NON_FEATURES = {'id', 'price_numeric', 'log_price',
                'estimated_revenue_l365d', 'estimated_occupancy_l365d',
                'month_label'}   # added in plot 2, not a model feature

FEATURE_COLS = [c for c in sf.columns if c not in NON_FEATURES]

model = xgb.XGBRegressor()
model.load_model(str(DATA / 'xgb_model.json'))

test = sf[sf['month'] == 11].copy()
pred_log   = model.predict(test[FEATURE_COLS])
pred_price = np.expm1(pred_log)
true_price = test['price_numeric'].values

# Cap display at $1000 for readability (99th pct ≈ $926 after cap)
mask = (true_price <= 1000) & (pred_price <= 1000)
t = true_price[mask]; p = pred_price[mask]
abs_err = np.abs(t - p)
pct_err  = abs_err / t * 100

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t, y=p,
    mode='markers',
    marker=dict(
        color=pct_err,
        colorscale='RdYlGn_r',
        cmin=0, cmax=40,
        size=4, opacity=0.5,
        colorbar=dict(title='% error', ticksuffix='%'),
    ),
    text=[f'True: ${tv:.0f}<br>Pred: ${pv:.0f}<br>Err: {e:.0f}%'
          for tv, pv, e in zip(t, p, pct_err)],
    hovertemplate='%{text}<extra></extra>',
    name='Listings',
))

lim = 1000
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim],
    mode='lines',
    line=dict(color='black', dash='dash', width=1),
    name='Perfect prediction',
    hoverinfo='skip',
))

fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim * 1.2],
    mode='lines', line=dict(color=GRAY, dash='dot', width=1),
    name='±20% band', showlegend=True, hoverinfo='skip',
))
fig.add_trace(go.Scatter(
    x=[0, lim], y=[0, lim * 0.8],
    mode='lines', line=dict(color=GRAY, dash='dot', width=1),
    showlegend=False, hoverinfo='skip',
))

fig.update_layout(
    title=dict(text='Predicted vs Actual Price — November 2025 Hold-Out Test',
               font_size=16),
    xaxis_title='Actual nightly price ($)',
    yaxis_title='Predicted nightly price ($)',
    xaxis=dict(range=[0, lim], tickprefix='$'),
    yaxis=dict(range=[0, lim * 1.1], tickprefix='$'),
    legend=dict(x=0.02, y=0.98),
)

fig.add_annotation(
    x=750, y=150,
    text=f'R² = 0.915<br>MdAPE = 10.4%<br>77% within ±20%<br>n = {mask.sum():,}',
    showarrow=False,
    bordercolor=GRAY, borderwidth=1, bgcolor='white',
    font_size=12, align='left',
)

save(fig, '06_pred_vs_actual')
fig.show()

  saved: plots/06_pred_vs_actual.png + .html


---
## Summary

In [8]:
plots = list(PLOTS.glob('*.png'))
print(f'Generated {len(plots)} plots in plots/:')
for p in sorted(plots):
    size_kb = p.stat().st_size // 1024
    print(f'  {p.name:<40} {size_kb} KB')


Generated 6 plots in plots/:
  01_seasonal_index.png                    120 KB
  02_price_by_month.png                    157 KB
  03_listing_map.png                       1700 KB
  04_price_by_borough_roomtype.png         135 KB
  05_shap_importance.png                   183 KB
  06_pred_vs_actual.png                    414 KB
